In [23]:
#=================================================================
# Celda 1 — Setup + inspeccionar JSONs disponibles
#=================================================================
import pandas as pd
import numpy as np
from pathlib import Path
import os
os.chdir("/workspaces/TFG_Spain-s_Migratory_Flow")

# ── Rutas siguiendo el patrón del proyecto ───────────────────
RAW_DIR    = Path("output/economico/01-raw")
SILVER_DIR = Path("output/economico/02-silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

# Cargar parquets generados por 01_economico
parquets = sorted(RAW_DIR.glob("*.parquet"))
assert len(parquets) > 0, f"❌ No hay parquets en {RAW_DIR}"

print(f"📂 Parquets encontrados:")
for p in parquets:
    print(f"   {p.name}")

# Cargar todos en un dict por nombre de tabla
dfs_raw = {}
for p in parquets:
    nombre = p.stem  # nombre sin extensión = nombre de tabla
    dfs_raw[nombre] = pd.read_parquet(p)
    print(f"✅ {nombre}: {dfs_raw[nombre].shape}")

📂 Parquets encontrados:
   raw_centros_educativos_ccaa.parquet
   raw_compraventas_vivienda.parquet
   raw_densidad_poblacion.parquet
   raw_hipotecas_vivienda.parquet
   raw_ipv_precio_vivienda_anual.parquet
   raw_ipv_precio_vivienda_ccaa.parquet
   raw_renta_media_hogar.parquet
   raw_tasa_paro_ccaa.parquet
   raw_turismo_pernoctaciones.parquet
   raw_turismo_viajeros.parquet
✅ raw_centros_educativos_ccaa: (180, 9)
✅ raw_compraventas_vivienda: (82440, 14)
✅ raw_densidad_poblacion: (3975, 12)
✅ raw_hipotecas_vivienda: (4460, 12)
✅ raw_ipv_precio_vivienda_anual: (2128, 11)
✅ raw_ipv_precio_vivienda_ccaa: (17024, 11)
✅ raw_renta_media_hogar: (576, 12)
✅ raw_tasa_paro_ccaa: (5040, 10)
✅ raw_turismo_pernoctaciones: (131670, 13)
✅ raw_turismo_viajeros: (136910, 14)


In [24]:
#=================================================================
# Celda 2 — Parser genérico de tablas INE
#=================================================================
def parse_serie_ine(serie: dict) -> list:
    """
    Convierte una serie INE {COD, Nombre, MetaData, Data} en lista de filas planas.
    MetaData → variables descriptivas (provincia, tipo, etc.)
    Data     → valores temporales {Fecha, Valor, Anyo}
    """
    meta = {}
    for m in serie.get("MetaData", []):
        var = m.get("T3_Variable", m.get("Variable", {}).get("Nombre", "var"))
        val = m.get("Nombre", "")
        meta[var] = val

    rows = []
    for d in serie.get("Data", []):
        row = {**meta}
        row["valor"]   = d.get("Valor")
        row["fecha"]   = d.get("Fecha", "")
        row["anyo"]    = d.get("Anyo",  str(row["fecha"])[:4] if row["fecha"] else None)
        row["secreto"] = d.get("Secreto", False)
        rows.append(row)
    return rows

def parse_tabla(registros: list) -> pd.DataFrame:
    rows = []
    for serie in registros:
        rows.extend(parse_serie_ine(serie))
    return pd.DataFrame(rows)

# Test rápido
for nombre, df in dfs_raw.items():
    print(f"  {nombre}: {df.shape} | cols: {list(df.columns)}")

  raw_centros_educativos_ccaa: (180, 9) | cols: ['total_nacional', 'nivel_educativo', 'valor', 'fecha', 'anyo', 'secreto', 'comunidades_y_ciudades_autónomas', 'fuente_tabla', 'fecha_captura']
  raw_compraventas_vivienda: (82440, 14) | cols: ['total_nacional', 'estado_de_la_vivienda', 'título_de_adquisición', 'tipo_de_dato', 'valor', 'fecha', 'anyo', 'secreto', 'trimestre', 'régimen_de_la_vivienda', 'comunidades_y_ciudades_autónomas', 'provincias', 'fuente_tabla', 'fecha_captura']
  raw_densidad_poblacion: (3975, 12) | cols: ['comunidades_y_ciudades_autónomas', 'sexo', 'tamaño_de_los_municipios', 'tipo_de_dato', 'valor', 'fecha', 'anyo', 'secreto', 'trimestre', 'provincias', 'fuente_tabla', 'fecha_captura']
  raw_hipotecas_vivienda: (4460, 12) | cols: ['naturaleza_de_la_finca', 'concepto_financiero', 'total_nacional', 'tipo_de_muestreo', 'valor', 'fecha', 'anyo', 'secreto', 'trimestre', 'comunidades_y_ciudades_autónomas', 'fuente_tabla', 'fecha_captura']
  raw_ipv_precio_vivienda_anual:

In [25]:
#=================================================================
# Celda 3 — Limpiar tasa de paro por provincia × año
#=================================================================
df_paro_raw = dfs_raw["raw_tasa_paro_ccaa"].copy()
print(df_paro_raw.head(3).to_string())
print(f"\nColumnas: {list(df_paro_raw.columns)}")
print(f"Tipos de dato únicos: {df_paro_raw.get('Tipo de dato', pd.Series()).unique()[:5]}")
print(f"Sexo únicos: {df_paro_raw.get('Sexo', pd.Series()).unique()}")

                          dim_0        dim_1           dim_2  dim_3  valor periodo                          fecha  secreto    fuente_tabla               fecha_captura
0  Tasa de paro de la población  Ambos sexos  Total Nacional  Total  11.76    None  2023-10-01T00:00:00.000+02:00    False  tasa_paro_ccaa  2026-03-25T10:35:35.016107
1  Tasa de paro de la población  Ambos sexos  Total Nacional  Total  11.84    None  2023-07-01T00:00:00.000+02:00    False  tasa_paro_ccaa  2026-03-25T10:35:35.016107
2  Tasa de paro de la población  Ambos sexos  Total Nacional  Total  11.60    None  2023-04-01T00:00:00.000+02:00    False  tasa_paro_ccaa  2026-03-25T10:35:35.016107

Columnas: ['dim_0', 'dim_1', 'dim_2', 'dim_3', 'valor', 'periodo', 'fecha', 'secreto', 'fuente_tabla', 'fecha_captura']
Tipos de dato únicos: []
Sexo únicos: []


In [26]:
#=================================================================
# Celda 4 — Limpiar tasa de paro (esquema dim_0..dim_3)
#=================================================================
df_paro = dfs_raw["raw_tasa_paro_ccaa"].copy()

df_paro_clean = (df_paro
    .rename(columns={
        "dim_0": "tipo_dato",
        "dim_1": "sexo",
        "dim_2": "Provincia",
        "dim_3": "nacionalidad",
    })
    .assign(
        año=lambda x: pd.to_numeric(
            x["fecha"].str[:4], errors="coerce"
        ).astype("Int64")
    )
    .query(
        "tipo_dato.str.contains('paro', case=False, na=False) and "
        "sexo == 'Ambos sexos' and "
        "Provincia != 'Total Nacional' and "
        "Provincia == Provincia and "
        "secreto == False"
    )
    .dropna(subset=["valor"])
    .groupby(["Provincia", "año"], as_index=False)["valor"]
    .mean()
    .rename(columns={"valor": "tasa_paro"})
    .dropna(subset=["Provincia", "año"])
)

print(f"✅ tasa_paro: {df_paro_clean.shape}")
print(f"   Años: {sorted(df_paro_clean['año'].dropna().unique())}")
print(f"   Provincias: {df_paro_clean['Provincia'].nunique()}")
print(df_paro_clean.head())

✅ tasa_paro: (57, 3)
   Años: [np.int64(2021), np.int64(2022), np.int64(2023)]
   Provincias: 19
   Provincia   año  tasa_paro
0  Andalucía  2021   33.20500
1  Andalucía  2022   27.18750
2  Andalucía  2023   28.75750
3     Aragón  2021   17.57875
4     Aragón  2022   16.65250


In [27]:
#=================================================================
# Celda 5 — Renta media por provincia × año
#=================================================================
# raw_renta_media_hogar solo tiene Total Nacional → usar municipio
df_renta = dfs_raw["raw_renta_neta_media_municipio"].copy()

print("Columnas:", list(df_renta.columns))
print(df_renta.head(2).to_string())
print()

# Filtrar solo provincias (excluir municipios y secciones/distritos)
# La col 'municipios' tiene valores tipo "01 Álava", "Vitoria-Gasteiz", etc.
# Las provincias suelen ser solo 2 dígitos numéricos al inicio
df_renta_clean = (df_renta
    .rename(columns={"municipios": "Provincia"})
    .assign(
        año=lambda x: pd.to_numeric(x["anyo"], errors="coerce").astype("Int64"),
        es_provincia=lambda x: x["Provincia"].str.match(r"^\d{2}\s", na=False)
    )
    .query("es_provincia == True and secreto == False")
    .dropna(subset=["valor"])
    .assign(Provincia=lambda x: x["Provincia"].str.replace(r"^\d{2}\s+", "", regex=True).str.strip())
    .groupby(["Provincia", "año"], as_index=False)["valor"]
    .mean()
    .rename(columns={"valor": "renta_media"})
    .dropna(subset=["Provincia", "año"])
    .drop(columns=["es_provincia"], errors="ignore")
)

print(f"\n✅ renta_media: {df_renta_clean.shape}")
print(f"   Años: {sorted(df_renta_clean['año'].dropna().unique())}")
print(f"   Provincias: {df_renta_clean['Provincia'].nunique()}")
print(df_renta_clean.head())

KeyError: 'raw_renta_neta_media_municipio'

In [ ]:
#=================================================================
# Celda 6 — IPV precio vivienda (nacional)
#=================================================================
df_ipv = dfs_raw["raw_ipv_precio_vivienda_anual"].copy()

print("Columnas IPV:", list(df_ipv.columns))
print(df_ipv.head(2).to_string())
print()

df_ipv_clean = (df_ipv
    .assign(año=lambda x: pd.to_numeric(x["fecha"].str[:4], errors="coerce").astype("Int64"))
    .query("secreto == False")
    .dropna(subset=["valor"])
    [["año", "valor"]]
    .groupby("año", as_index=False)["valor"]
    .mean()
    .rename(columns={"valor": "ipv_vivienda"})
)

print(f"✅ ipv_vivienda: {df_ipv_clean.shape}")
print(df_ipv_clean.head())

✅ ipv_vivienda (nacional): (19, 2)
    año  ipv_vivienda
0  2007     80.635545
1  2008     74.975741
2  2009     67.926589
3  2010     67.936268
4  2011     60.288830


In [ ]:
#=================================================================
# Celda 7 — Merge final provincia × año + normalización
#=================================================================
from functools import reduce

# Merge tasa_paro + renta (ambas por provincia×año)
df_eco = pd.merge(df_paro_clean, df_renta_clean, on=["Provincia", "año"], how="outer")

# IPV es nacional → join por año
df_eco = pd.merge(df_eco, df_ipv_clean, on="año", how="left")

# Normalizar nombre provincia
df_eco["Provincia"] = df_eco["Provincia"].str.strip().str.title()
df_eco["año"] = df_eco["año"].astype("Int64")

# Ordenar
df_eco = df_eco.sort_values(["Provincia", "año"]).reset_index(drop=True)

print(f"📊 Panel económico final: {df_eco.shape}")
print(f"   Años:       {sorted(df_eco['año'].dropna().unique())}")
print(f"   Provincias: {df_eco['Provincia'].nunique()}")
print(f"\n   Nulos por columna:\n{df_eco.isnull().sum()}")
print(f"\n{df_eco.head(12).to_string()}")

📊 Panel económico final: (1248, 5)
   Años:       [np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
   Provincias: 52

   Nulos por columna:
Provincia          0
año                0
tasa_paro          0
renta_media     1248
ipv_vivienda     260
dtype: int64

   Provincia   año  tasa_paro  renta_media  ipv_vivienda
0   Albacete  2002     6.9350          NaN           NaN
1   Albacete  2003     7.7000          NaN           NaN
2   Albacete  2004     9.0050          NaN           NaN
3   Albacete  2005     9.8400          NaN           NaN
4   Albacete  2006    10.0150          NaN           NaN
5   Albacete  2007     9.1625          NaN     80.635545
6   Al

In [ ]:
#=================================================================
# Celda 8 — Export (Guardado)
#=================================================================
ruta = SILVER_DIR / "economico_provincia_año.parquet"
df_eco.to_parquet(ruta, index=False)

# También CSV por compatibilidad con merge maestro
df_eco.to_csv(SILVER_DIR / "economico_provincia_año.csv", index=False)

print(f"✅ Guardado en: {SILVER_DIR}")
print(f"   Shape final: {df_eco.shape}")



✅ Guardado en: output/economico/02-silver
   Shape final: (1248, 5)
